# 07c · Önceden kayıtlı test: Vadeli yön uyumu
## 1 dakikalık veri, 2016–2026

Bu not defteri **KARARLAR.md → "Önceden kayıtlı test: Vadeli yön uyumu (07c)"** bölümündeki kuralı **birebir** uygular. Kural 15/11/2026'da, bu veri indirilmeden önce kayda geçirildi (commit `2cc9382`). Buradaki hiçbir eşik veriyi gördükten sonra değiştirilemez; değiştirilirse test geçersiz olur.

**Hipotez (07b'de 24/08–13/11/2026 verisinde bulundu):** ORB işlemi, endeks vadelisi zaten işlem yönünde hareket ediyorsa daha iyi sonuç verir.

**Vadeli yön uyumu skoru (0–4)** — her işlem için işlem yönünde olan koşul sayısı:
1. Eşleşen vadelinin önceki seans kapanışından 09:30'a getirisi
2. Vadelinin 09:30 fiyatının gece aralığının hangi yarısında olduğu
3. ES ve NQ açılış aralığı mumunun ikisinin de yönü
4. Girişten **önce tamamlanmış** mumlarda gece yükseği (uzun) / düşüğü (kısa) aşılmış mı

**Kural:** skor ≥ 3 → işlem al. **Doğrulandı** sayılmak için her aralıkta (5 dk ve 15 dk) üç şart: (1) gün bazında permütasyonla tek yönlü p < 0,025, (2) fark 2016–2020 ve 2021–2026'da ayrı ayrı pozitif, (3) fark iyimser mum içi varsayımda da pozitif.

**Veri:** Alpaca (ETF ve hisseler, 1 dk, tüm borsalar) ve Databento (ES, NQ, RTY, 1 dk, gece dahil). Anahtarlar Colab **Gizli anahtarlar**'dan okunur: `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, `DATABENTO_API_KEY`.

**Maliyet:** Alpaca ücretsiz. Databento indirmeden önce maliyeti hesaplar; **40 $'ı aşarsa durur**. İndirilen veri önbelleğe yazılır; ikinci çalıştırmada tekrar ücret ödenmez (Google Drive önbelleği önerilir).

**Süre:** İlk çalıştırmada Alpaca indirmesi 10–15 dakika sürebilir. Sonraki çalıştırmalar önbellekten gelir.

> ⚠️ Öğrenme amaçlıdır, yatırım tavsiyesi değildir.

In [ ]:
#@title ⚙️ Ayarlar
ONBELLEK = "Google Drive" #@param ["Google Drive", "Colab oturumu"]
#@markdown Google Drive: veri kalıcı saklanır, Databento'ya ikinci kez ödeme yapılmaz (Drive bağlama izni istenir).
PERMUTASYON_TEKRAR = 5000 #@param {type:"integer"}
RASTGELE_TEKRAR = 20 #@param {type:"integer"}
#@markdown ---
#@markdown **Aşağıdakiler KARARLAR.md'de sabittir — değiştirme.**
TEST_BASLANGIC = "2016-01-01"
TEST_BITIS = "2026-08-21"          # keşif dönemi (24/08–13/11/2026) hariç
MALIYET_SINIRI_USD = 40.0
ETFLER = ["SPY", "QQQ", "IWM"]
MAG7 = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]
VADELI_ESLEME = {"SPY": "ES", "QQQ": "NQ", "IWM": "RTY"}   # Mag7 -> NQ
KOMISYON_HISSE = 0.0035
KAYMA_BPS = {"ETF": 1, "Mag7": 3}
SON_GIRIS = "15:30"
SKOR_ESIGI = 3
ALFA = 0.025

## 1 · Kurulum, anahtarlar, önbellek

In [ ]:
import subprocess, sys, os, time, json
for paket in ["databento", "pyarrow"]:
    try:
        __import__(paket)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket], check=True)
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def tr_sayi(x, ond=2):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    s = f"{abs(x):,.{ond}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("-" if x < 0 else "") + s

def tr_yuzde(x, ond=1):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    return ("-" if x < 0 else "") + "%" + tr_sayi(abs(x) * 100, ond)

def tr_tarih(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")

def dk(saat):
    s, d = saat.split(":")
    return int(s) * 60 + int(d)

def tr_tablo(t, tam=("işlem", "n", "gün"), yuzde=("isabet", "pay"), ond=3):
    b = t.copy().astype(object)
    for col in t.columns:
        for idx in t.index:
            x = t.at[idx, col]
            if isinstance(x, str):
                continue
            if col in tam:
                b.at[idx, col] = tr_sayi(x, 0)
            elif any(str(col).startswith(y) for y in yuzde):
                b.at[idx, col] = tr_yuzde(x)
            else:
                b.at[idx, col] = tr_sayi(x, ond)
    return b

from google.colab import userdata
ANAHTAR = {}
for ad in ["ALPACA_API_KEY", "ALPACA_SECRET_KEY", "DATABENTO_API_KEY"]:
    try:
        ANAHTAR[ad] = userdata.get(ad)
        print(f"✅ {ad} okundu")
    except Exception as e:
        raise RuntimeError(f"{ad} okunamadı. Sol menüdeki anahtar simgesinden ekle ve 'Not defteri erişimi'ni aç. ({e})")

if ONBELLEK == "Google Drive":
    from google.colab import drive
    drive.mount("/content/drive")
    KLASOR = "/content/drive/MyDrive/abd-borsa-calismalari/veri07c"
else:
    KLASOR = "/content/veri07c"
os.makedirs(KLASOR, exist_ok=True)
print("Önbellek klasörü:", KLASOR)

SEMBOLLER = ETFLER + MAG7
GRUP = {s: ("ETF" if s in ETFLER else "Mag7") for s in SEMBOLLER}
VADELI = {s: VADELI_ESLEME.get(s, "NQ") for s in SEMBOLLER}
ACILIS = dk("09:30")
rng = np.random.default_rng(42)

## 2 · Veri indirme
**Alpaca:** 1 dakikalık mumlar, düzeltilmemiş fiyat (ORB gün içi olduğu için bölünme düzeltmesi gerekmez), tüm borsalar (`feed=sip`). Yalnızca seans (09:30–16:00 ET) tutulur.

**Sembol değişikliği:** META 2022'den önce FB olarak işlem görüyordu. META için 2022 öncesi veri gelmezse FB'den tamamlanır.

**Databento:** CME Globex, 1 dk OHLCV, sürekli kontrat (`ES.c.0`, `NQ.c.0`, `RTY.c.0`). Her mumda kontrat kimliği (`instrument_id`) gelir; kontrat değişimi olan günler aşağıda ayıklanır. RTY CME'de 2017 ortasında işlem görmeye başladı; öncesinde IWM günleri vadeli verisi olmadığı için testten çıkar (kayıttaki kural).

In [ ]:
ALPACA_URL = "https://data.alpaca.markets/v2/stocks/bars"

def alpaca_indir(sembol, baslangic, bitis):
    basliklar = {"APCA-API-KEY-ID": ANAHTAR["ALPACA_API_KEY"], "APCA-API-SECRET-KEY": ANAHTAR["ALPACA_SECRET_KEY"]}
    parametre = {"symbols": sembol, "timeframe": "1Min", "start": f"{baslangic}T00:00:00Z",
                 "end": f"{bitis}T23:59:00Z", "limit": 10000, "adjustment": "raw", "feed": "sip", "sort": "asc"}
    satirlar, sayfa = [], None
    while True:
        if sayfa:
            parametre["page_token"] = sayfa
        for deneme in range(5):
            y = requests.get(ALPACA_URL, headers=basliklar, params=parametre, timeout=60)
            if y.status_code == 429:
                time.sleep(5 * (deneme + 1)); continue
            y.raise_for_status(); break
        j = y.json()
        satirlar.extend(j.get("bars", {}).get(sembol, []) or [])
        sayfa = j.get("next_page_token")
        time.sleep(0.31)                      # ücretsiz plan: dakikada 200 istek
        if not sayfa:
            break
    if not satirlar:
        return pd.DataFrame()
    df = pd.DataFrame(satirlar)[["t", "o", "h", "l", "c", "v"]]
    df.index = pd.to_datetime(df.pop("t"), utc=True).dt.tz_convert("America/New_York")
    dakika = df.index.hour * 60 + df.index.minute
    return df[(dakika >= ACILIS) & (dakika < dk("16:00"))].astype("float64")

def databento_indir(kokler, baslangic, bitis):
    import databento as dbn
    istemci = dbn.Historical(ANAHTAR["DATABENTO_API_KEY"])
    sorgu = dict(dataset="GLBX.MDP3", symbols=[f"{k}.c.0" for k in kokler], schema="ohlcv-1m",
                 stype_in="continuous", start=f"{baslangic}T00:00:00", end=f"{bitis}T23:59:00")
    maliyet = istemci.metadata.get_cost(**sorgu)
    print(f"Databento tahmini maliyet: {tr_sayi(maliyet, 2)} $ (sınır {tr_sayi(MALIYET_SINIRI_USD, 0)} $)")
    if maliyet > MALIYET_SINIRI_USD:
        raise RuntimeError("Maliyet sınırı aşıldı; indirme yapılmadı. KARARLAR.md'deki sınırı birlikte gözden geçirelim.")
    df = istemci.timeseries.get_range(**sorgu).to_df()
    df.index = df.index.tz_convert("America/New_York")
    return df[["symbol", "instrument_id", "open", "high", "low", "close", "volume"]]

In [ ]:
def onbellekten(ad, uret):
    yol = os.path.join(KLASOR, f"{ad}.parquet")
    if os.path.exists(yol):
        return pd.read_parquet(yol)
    df = uret()
    if len(df):
        df.to_parquet(yol)
    return df

BAR = {}
for s in SEMBOLLER:
    t0 = time.time()
    df = onbellekten(f"alpaca_{s}", lambda: alpaca_indir(s, TEST_BASLANGIC, TEST_BITIS))
    if s == "META" and (df.empty or df.index.min() > pd.Timestamp("2016-02-01", tz="America/New_York")):
        bitis_fb = (df.index.min() - pd.Timedelta(days=1)).strftime("%Y-%m-%d") if len(df) else TEST_BITIS
        fb = onbellekten("alpaca_FB", lambda: alpaca_indir("FB", TEST_BASLANGIC, bitis_fb))
        df = pd.concat([fb, df]).sort_index()
        df = df[~df.index.duplicated()]
        print("META: 2022 öncesi FB sembolünden tamamlandı.")
    BAR[s] = df
    gunler = pd.Series(df.index.date).nunique() if len(df) else 0
    print(f"{s:5s}: {tr_sayi(gunler, 0)} gün, {tr_sayi(len(df), 0)} mum ({tr_sayi(time.time() - t0, 0)} sn)")

VADELI_HAM = onbellekten("databento_ES_NQ_RTY", lambda: databento_indir(["ES", "NQ", "RTY"], TEST_BASLANGIC, TEST_BITIS))

def vadeli_secimi(kok):
    """Sembol sütunu 'ES.c.0' ya da gerçek kontrat adı ('ESZ5') olabilir; ikisini de tanır."""
    sembol = VADELI_HAM["symbol"].astype(str)
    return VADELI_HAM[(sembol == f"{kok}.c.0") | sembol.str.match(rf"^{kok}[FGHJKMNQUVXZ]\d")]

for k in ["ES", "NQ", "RTY"]:
    alt = vadeli_secimi(k)
    print(f"{k:3s}: {tr_sayi(len(alt), 0)} mum, {tr_tarih(alt.index.min())} – {tr_tarih(alt.index.max())}")

## 3 · Veri kalite kontrolü
- **Yıllara göre gün sayısı:** her yıl ~250 seans beklenir. 2021'e özellikle bakılır (Alpaca'nın eski duyurusunda o yıl için boşluk uyarısı vardı).
- **Eksik açılış:** 09:30 mumu olmayan günler testten çıkar.

In [ ]:
kalite = {}
for s in SEMBOLLER:
    d = BAR[s]
    if d.empty:
        continue
    gunluk = pd.Series(1, index=pd.Index(pd.to_datetime(d.index.date)).unique())
    kalite[s] = gunluk.groupby(gunluk.index.year).size()
kalite = pd.DataFrame(kalite).fillna(0).astype(int)
display(Markdown("**Yıllara göre seans günü sayısı**"))
display(kalite)
supheli = kalite[(kalite < 240).any(axis=1)]
if len(supheli):
    print("⚠️ 240'tan az günü olan yıllar var (2026 kısmi yıl olduğu için normal):", ", ".join(str(y) for y in supheli.index))

## 4 · Günlere bölme ve vadeli özellikleri
- **İşlem takvimi:** SPY'ın seans günleri.
- **Gece aralığı:** önceki seans günü 18:00 ET – bugün 09:30 ET (Pazartesi için Cuma 18:00).
- **Kontrat değişimi:** önceki kapanış, gece aralığı ve bugünün seansı aynı kontrata ait değilse o gün o vadeli için **eksik** sayılır (getiri sahte bir sıçrama içerir).

In [ ]:
def gunlere_bol(df):
    gunler = {}
    if df.empty:
        return gunler
    ts = df.index
    tarih = pd.to_datetime(ts.date)
    dakika = (ts.hour * 60 + ts.minute).values
    o, h, l, c = (df[k].values for k in ["o", "h", "l", "c"])
    sinir = np.flatnonzero(np.r_[True, tarih[1:] != tarih[:-1], True])
    for a, b in zip(sinir[:-1], sinir[1:]):
        if dakika[a] != ACILIS:
            continue
        gunler[tarih[a]] = dict(dk=dakika[a:b], o=o[a:b], h=h[a:b], l=l[a:b], c=c[a:b])
    return gunler

GUN = {s: gunlere_bol(BAR[s]) for s in SEMBOLLER}
TAKVIM = sorted(GUN["SPY"].keys())
ONCEKI = {TAKVIM[i]: TAKVIM[i - 1] for i in range(1, len(TAKVIM))}

def vadeli_gunleri(kok):
    d = vadeli_secimi(kok).sort_index()
    ts = d.index.as_unit("ns").asi8              # birim: nanosaniye (pandas sürümünden bağımsız)
    ny = lambda gun, saat: pd.Timestamp(gun).tz_localize("America/New_York").value + int(saat * 3600e9)
    o, h, l, c, kimlik = (d[k].values for k in ["open", "high", "low", "close", "instrument_id"])
    sonuc = {}
    for gun, dun in ONCEKI.items():
        g_bas, g_bit = np.searchsorted(ts, [ny(dun, 18), ny(gun, 9.5)])
        s_bas, s_bit = np.searchsorted(ts, [ny(gun, 9.5), ny(gun, 16)])
        k_bas, k_bit = np.searchsorted(ts, [ny(dun, 9.5), ny(dun, 16)])
        if g_bit - g_bas < 60 or s_bit - s_bas < 30 or k_bit == k_bas:
            continue
        kimlikler = np.unique(np.r_[kimlik[k_bit - 1], kimlik[g_bas:g_bit], kimlik[s_bas:s_bit]])
        if len(kimlikler) != 1:
            continue                                   # kontrat değişimi: eksik gün
        dakika = ((ts[s_bas:s_bit] - ny(gun, 0)) // 60_000_000_000).astype(int)
        sonuc[gun] = dict(onh=h[g_bas:g_bit].max(), onl=l[g_bas:g_bit].min(), p0930=c[g_bit - 1],
                          onceki_kapanis=c[k_bit - 1], dk=dakika, o=o[s_bas:s_bit], h=h[s_bas:s_bit],
                          l=l[s_bas:s_bit], c=c[s_bas:s_bit])
    return sonuc

VGUN = {k: vadeli_gunleri(k) for k in ["ES", "NQ", "RTY"]}
for k, v in VGUN.items():
    print(f"{k:3s}: kullanılabilir {tr_sayi(len(v), 0)} gün / takvim {tr_sayi(len(ONCEKI), 0)} gün")
    if len(v) == 0:
        raise RuntimeError(f"{k}: hiç kullanılabilir gün yok — veri ya da zaman dilimi sorunu, test durduruldu.")

## 5 · ORB motoru (1 dk) ve skor
Kayıttaki sabit kurallar: aralık yönü, aralık ucuna anlık stop emri, stop aralığın karşı ucu, gün sonunda çıkış, son giriş 15:30. Giriş mumunda stopa değmenin sırası bilinmediği için **kötümser** (değdiyse stop) esastır; **iyimser** (yalnızca kapanış stopun ötesindeyse stop) ayrıca hesaplanır.

In [ ]:
def orb(g, or_dk, yon_zorla=None):
    dkk, o, h, l, c = g["dk"], g["o"], g["h"], g["l"], g["c"]
    or_maske = dkk < ACILIS + or_dk
    k = int(or_maske.sum())
    if k < max(1, or_dk // 2) or k >= len(o):
        return None
    orh, orl, oro, orc = h[:k].max(), l[:k].min(), o[0], c[k - 1]
    if abs(orc - oro) < 1e-4 * oro:
        return None
    yon = yon_zorla if yon_zorla is not None else (1 if orc > oro else -1)
    seviye, stop = (orh, orl) if yon == 1 else (orl, orh)
    aday = np.arange(k, len(o))
    aday = aday[dkk[aday] <= dk(SON_GIRIS)]
    vurdu = (h[aday] >= seviye) if yon == 1 else (l[aday] <= seviye)
    if not vurdu.any():
        return None
    e = aday[np.argmax(vurdu)]
    giris = max(seviye, o[e]) if yon == 1 else min(seviye, o[e])
    risk = (giris - stop) * yon
    if risk <= 0:
        return None
    sonuc = {}
    for varsayim in ["k", "i"]:
        cikis = None
        degdi = (l[e] <= stop) if yon == 1 else (h[e] >= stop)
        kesin = (c[e] <= stop) if yon == 1 else (c[e] >= stop)
        if (varsayim == "k" and degdi) or (varsayim == "i" and kesin):
            cikis = stop
        else:
            sonra = np.arange(e + 1, len(o))
            st = (l[sonra] <= stop) if yon == 1 else (h[sonra] >= stop)
            if st.any():
                i = sonra[np.argmax(st)]
                cikis = min(o[i], stop) if yon == 1 else max(o[i], stop)
            else:
                cikis = c[-1]
        sonuc[varsayim] = cikis
    return dict(yon=yon, giris=giris, stop=stop, risk=risk, giris_dk=int(dkk[e]),
                cikis_k=sonuc["k"], cikis_i=sonuc["i"])

def vadeli_or_yonu(kok, gun, or_dk):
    v = VGUN[kok].get(gun)
    if v is None:
        return np.nan
    m = v["dk"] < ACILIS + or_dk
    if m.sum() == 0 or v["dk"][0] != ACILIS:
        return np.nan
    return np.sign(v["c"][m][-1] - v["o"][0])

def skor(sembol, gun, r, or_dk):
    v = VGUN[VADELI[sembol]].get(gun)
    if v is None:
        return np.nan, [np.nan] * 4
    yon = r["yon"]
    k1 = float(np.sign(v["p0930"] - v["onceki_kapanis"]) == yon)
    orta = (v["onh"] + v["onl"]) / 2
    k2 = float((v["p0930"] > orta) if yon == 1 else (v["p0930"] < orta))
    es, nq = vadeli_or_yonu("ES", gun, or_dk), vadeli_or_yonu("NQ", gun, or_dk)
    if np.isnan(es) or np.isnan(nq):
        return np.nan, [k1, k2, np.nan, np.nan]
    k3 = float(es == yon and nq == yon)
    once = (v["dk"] >= ACILIS) & (v["dk"] < r["giris_dk"])      # girişten önce tamamlanmış mumlar
    if not once.any():
        return np.nan, [k1, k2, k3, np.nan]
    k4 = float(v["h"][once].max() >= v["onh"]) if yon == 1 else float(v["l"][once].min() <= v["onl"])
    return k1 + k2 + k3 + k4, [k1, k2, k3, k4]

def islemler(or_dk, yon_uretici=None, skorlu=True):
    satir = []
    for s in SEMBOLLER:
        kayma = KAYMA_BPS[GRUP[s]]
        for gun, g in GUN[s].items():
            if gun not in ONCEKI:
                continue
            r = orb(g, or_dk, yon_uretici() if yon_uretici else None)
            if r is None:
                continue
            maliyet = lambda cikis: 2 * KOMISYON_HISSE + kayma / 1e4 * (r["giris"] + cikis)
            kayit = dict(sembol=s, grup=GRUP[s], tarih=gun, yon=r["yon"],
                         R_k=((r["cikis_k"] - r["giris"]) * r["yon"] - maliyet(r["cikis_k"])) / r["risk"],
                         R_i=((r["cikis_i"] - r["giris"]) * r["yon"] - maliyet(r["cikis_i"])) / r["risk"])
            if skorlu:
                sk, parca = skor(s, gun, r, or_dk)
                kayit.update(skor=sk, k1=parca[0], k2=parca[1], k3=parca[2], k4=parca[3])
            satir.append(kayit)
    return pd.DataFrame(satir)

ISLEM = {}
for or_dk in [5, 15]:
    t0 = time.time()
    df = islemler(or_dk)
    ISLEM[or_dk] = df
    print(f"{or_dk} dk: {tr_sayi(len(df), 0)} işlem, skoru hesaplanabilen {tr_sayi(int(df['skor'].notna().sum()), 0)} "
          f"({tr_yuzde(df['skor'].notna().mean())}) · {tr_sayi(time.time() - t0, 0)} sn")

## 6 · Önceden kayıtlı test
**Fark** = "skor ≥ 3" işlemlerinin ortalama net R'ı − "skor ≤ 2" işlemlerinin ortalaması (kötümser varsayım).

**Gün bazında permütasyon:** aynı gündeki işlemler (ortak piyasa hareketi, ortak vadeli verisi) birlikte karıştırılır. Aynı sayıda işlemi olan günler kendi aralarında etiket dizilerini değiş tokuş eder; gün içindeki etiket sırası da karıştırılır. Tek yönlü p = karışık farkların gerçek farka eşit ya da büyük olma oranı.

In [ ]:
def permutasyon_p(df, tekrar):
    d = df.dropna(subset=["skor"]).sort_values(["tarih", "sembol"])
    r = d["R_k"].values
    m = (d["skor"] >= SKOR_ESIGI).values
    gozlenen = r[m].mean() - r[~m].mean()
    tarih = d["tarih"].values
    sinir = np.flatnonzero(np.r_[True, tarih[1:] != tarih[:-1], True])
    gunler = [(a, b) for a, b in zip(sinir[:-1], sinir[1:])]
    gruplar = {}
    for a, b in gunler:
        gruplar.setdefault(b - a, []).append(a)
    R_g, M_g = {}, {}
    for boy, baslar in gruplar.items():
        idx = np.array(baslar)[:, None] + np.arange(boy)[None, :]
        R_g[boy], M_g[boy] = r[idx], m[idx]
    toplam_r, n = r.sum(), len(r)
    say = 0
    for _ in range(tekrar):
        top_m, n_m = 0.0, 0
        for boy in R_g:
            Mp = M_g[boy][rng.permutation(len(M_g[boy]))]
            Mp = np.take_along_axis(Mp, rng.random(Mp.shape).argsort(axis=1), axis=1)
            top_m += (R_g[boy] * Mp).sum(); n_m += Mp.sum()
        if 0 < n_m < n:
            fark = top_m / n_m - (toplam_r - top_m) / (n - n_m)
            if fark >= gozlenen:
                say += 1
    return gozlenen, (say + 1) / (tekrar + 1)

def fark(d, kol="R_k"):
    d = d.dropna(subset=["skor"])
    m = d["skor"] >= SKOR_ESIGI
    return d.loc[m, kol].mean() - d.loc[~m, kol].mean() if m.any() and (~m).any() else np.nan

SONUC = []
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    t0 = time.time()
    f_k, p = permutasyon_p(d, PERMUTASYON_TEKRAR)
    yil = pd.to_datetime(d["tarih"]).dt.year
    f_a, f_b = fark(d[yil <= 2020]), fark(d[yil >= 2021])
    f_i = fark(d, "R_i")
    sartlar = [p < ALFA, f_a > 0 and f_b > 0, f_i > 0]
    SONUC.append({"aralık": f"{or_dk} dk", "işlem": int(d["skor"].notna().sum()),
                  "skor≥3 pay": (d["skor"] >= SKOR_ESIGI).sum() / d["skor"].notna().sum(),
                  "fark (kötümser)": f_k, "p": p, "fark 2016–2020": f_a, "fark 2021–2026": f_b,
                  "fark (iyimser)": f_i,
                  "şart 1 (p<0,025)": "✅" if sartlar[0] else "❌",
                  "şart 2 (iki dönem)": "✅" if sartlar[1] else "❌",
                  "şart 3 (iyimser)": "✅" if sartlar[2] else "❌",
                  "SONUÇ": "DOĞRULANDI" if all(sartlar) else "DOĞRULANMADI"})
    print(f"{or_dk} dk permütasyon: {time.time() - t0:.0f} sn")
SONUC = pd.DataFrame(SONUC).set_index("aralık")
display(Markdown("### Karar tablosu"))
display(tr_tablo(SONUC))

## 7 · Raporlanır ama karar vermez

In [ ]:
for or_dk in [5, 15]:
    d = ISLEM[or_dk].dropna(subset=["skor"])
    t = d.groupby("skor").agg(işlem=("R_k", "size"), isabet=("R_k", lambda x: (x > 0).mean()),
                              **{"net R kötümser": ("R_k", "mean"), "net R iyimser": ("R_i", "mean")})
    t.index = [f"skor {int(i)}" for i in t.index]
    display(Markdown(f"**{or_dk} dk · skor 0–4 boyunca sonuç**"))
    display(tr_tablo(t))

    satir = []
    for ad, alt in [("TÜMÜ", d)] + [(g, d[d["grup"] == g]) for g in ["ETF", "Mag7"]] + [(s, d[d["sembol"] == s]) for s in SEMBOLLER]:
        if alt.empty:
            continue
        m = alt["skor"] >= SKOR_ESIGI
        satir.append({"kırılım": ad, "işlem": len(alt), "skor≥3 işlem": int(m.sum()),
                      "skor≥3 net R": alt.loc[m, "R_k"].mean(), "skor≤2 net R": alt.loc[~m, "R_k"].mean(),
                      "filtresiz net R": alt["R_k"].mean(), "fark": fark(alt)})
    display(Markdown(f"**{or_dk} dk · grup ve sembol kırılımı (kötümser)**"))
    display(tr_tablo(pd.DataFrame(satir).set_index("kırılım"), tam=("işlem", "skor≥3 işlem")))

    yil = pd.to_datetime(d["tarih"]).dt.year
    yt = d.assign(yil=yil).groupby("yil").apply(
        lambda x: pd.Series({"işlem": len(x), "skor≥3 net R": x.loc[x["skor"] >= SKOR_ESIGI, "R_k"].mean(),
                             "skor≤2 net R": x.loc[x["skor"] < SKOR_ESIGI, "R_k"].mean(), "fark": fark(x)}),
        include_groups=False)
    display(Markdown(f"**{or_dk} dk · yıllara göre**"))
    display(tr_tablo(yt))

In [ ]:
#@title 🎲 Rastgele yönlü ORB ile kıyas (filtresiz, kötümser)
for or_dk in [5, 15]:
    ort = []
    for _ in range(RASTGELE_TEKRAR):
        ort.append(islemler(or_dk, yon_uretici=lambda: int(rng.choice([-1, 1])), skorlu=False)["R_k"].mean())
    d = ISLEM[or_dk]
    skorlu = d.dropna(subset=["skor"])
    print(f"{or_dk} dk · filtresiz strateji {tr_sayi(d['R_k'].mean(), 3)} R · rastgele yön ort. {tr_sayi(np.mean(ort), 3)} R "
          f"(aralık {tr_sayi(np.min(ort), 3)} / {tr_sayi(np.max(ort), 3)}) · skor≥3 {tr_sayi(skorlu.loc[skorlu['skor'] >= SKOR_ESIGI, 'R_k'].mean(), 3)} R")

In [ ]:
#@title 📈 Kümülatif net R: skor ≥ 3 ile filtresiz
fig, eksen = plt.subplots(1, 2, figsize=(12, 4))
for ax, or_dk in zip(eksen, [5, 15]):
    d = ISLEM[or_dk].dropna(subset=["skor"]).sort_values("tarih")
    g = d.groupby("tarih")
    ax.plot(pd.to_datetime(list(g.groups)), g["R_k"].sum().cumsum().values, label="filtresiz", lw=1, color="#9aa5b1")
    f = d[d["skor"] >= SKOR_ESIGI].groupby("tarih")["R_k"].sum().cumsum()
    ax.plot(pd.to_datetime(f.index), f.values, label="skor ≥ 3", lw=1.5, color="#c0392b")
    ax.axhline(0, color="grey", lw=0.6); ax.set_title(f"{or_dk} dk ORB · kümülatif net R (kötümser)")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8 · Kayıt ve X gönderisi
Karar tablosu KARARLAR.md'deki kayıtlı bölümün altına **değiştirilmeden** eklenecek.

In [ ]:
satirlar = []
for ar, r in SONUC.iterrows():
    satirlar.append(f"{ar}: {r['SONUÇ']} · fark {tr_sayi(r['fark (kötümser)'], 3)} R, p={tr_sayi(r['p'], 4)}, "
                    f"2016–2020 {tr_sayi(r['fark 2016–2020'], 3)} / 2021–2026 {tr_sayi(r['fark 2021–2026'], 3)}, "
                    f"iyimser {tr_sayi(r['fark (iyimser)'], 3)}, {tr_sayi(r['işlem'], 0)} işlem")
print("KARARLAR İÇİN SONUÇ SATIRLARI\n" + "-" * 40)
print("\n".join(satirlar))

dogru = [ar for ar, r in SONUC.iterrows() if r["SONUÇ"] == "DOĞRULANDI"]
gonderi = (
    "Önceden kayıtlı test: ORB + endeks vadelisi yön uyumu\n"
    "$SPY $QQQ $IWM + Mag7 · 2016–2026 · 1 dk veri\n\n"
    "Kuralı 2026'nın 60 gününde buldum, veriyi indirmeden önce yazıya döktüm, sonra 10 yılda test ettim.\n\n"
    + "\n".join(f"{ar}: {'doğrulandı' if r['SONUÇ'] == 'DOĞRULANDI' else 'doğrulanmadı'} (fark {tr_sayi(r['fark (kötümser)'], 2)} R, p={tr_sayi(r['p'], 3)})" for ar, r in SONUC.iterrows())
    + "\n\n" + ("Kısa veride bulunan etki uzun veride de tuttu." if dogru else "Kısa veride görünen etki uzun veride tutmadı. Kayıt bu yüzden önemli.")
    + "\n\nÖğrenme amaçlıdır, yatırım tavsiyesi değildir."
)
print("\nKOPYALAMAK İÇİN DÜZ METİN\n" + "-" * 40)
print(gonderi)
print("-" * 40, f"\n{len(gonderi)} karakter")